In [ ]:
%pip install boto3 pyyaml

In [33]:
import os
import json

#configure the environment Before network modules
os.environ["USE_LOCALSTACK"] = "1"
os.environ["LOCALSTACK_URL"] = "http://localhost:4566"

#Ensure state/state.json exists
os.makedirs("state", exist_ok=True)
if not  os.path.exists("state/state.json"):
    with open("state/state.json", "w") as f:
        json.dump({}, f)
print("Environment configured for Floci at :" , os.environ["LOCALSTACK_URL"])

Environment configured for Floci at : http://localhost:4566


In [34]:

from network.planner import load_config
from network.main import command_plan , command_apply, command_destroy
from network.ec2_instance import create_ec2_instance

config = load_config()
print("Loaded config for vpc:", config["vpc"]["name"])
print(config["security_groups"])

Loaded config for vpc: dev-vpc
[{'name': 'dev-web-sg', 'description': 'Allow SSH, HTTP, HTTPS inbound', 'ingress': [{'protocol': 'tcp', 'port': 22, 'cidr': '0.0.0.0/0', 'description': 'SSH access'}, {'protocol': 'tcp', 'port': 80, 'cidr': '0.0.0.0/0', 'description': 'HTTP'}, {'protocol': 'tcp', 'port': 443, 'cidr': '0.0.0.0/0', 'description': 'HTTPs'}], 'egress': [{'protocol': 'tcp', 'port': 80, 'cidr': '0.0.0.0/0', 'description': 'HTTP'}]}]


In [35]:
plan = command_plan(config)


 Scanning the  battlefiend that is config file
🔍 Verifying resources against AWS...

  ✅ No drift — state matches AWS.

  EXECUTION PLAN
  ⚡ vpc                  "dev-vpc" → NEEDS CREATION
  ⚡ subnet               "public-subnet-az1" → NEEDS CREATION
  ⚡ subnet               "public-subnet-az2" → NEEDS CREATION
  ⚡ subnet               "private-subnet-az1" → NEEDS CREATION
  ⚡ subnet               "private-subnet-az2" → NEEDS CREATION
  ⚡ internet_gateway     "dev-igw" → NEEDS CREATION
  ⚡ nat_gateway          "dev-nat" → NEEDS CREATION
  ⚡ route_table          "public-route-table" → NEEDS CREATION
  ⚡ route_table          "private-route-table" → NEEDS CREATION
  ⚡ security_group       "dev-web-sg" → NEEDS CREATION
  10 to create, 0 to delete, 0 unchanged

  ⚡ Run 'python main.py apply' to execute this plan.


In [36]:
command_apply(config)


 DEPLOYING - Building Infra........

🔍 Verifying resources against AWS...

  ✅ No drift — state matches AWS.

  EXECUTION PLAN
  ⚡ vpc                  "dev-vpc" → NEEDS CREATION
  ⚡ subnet               "public-subnet-az1" → NEEDS CREATION
  ⚡ subnet               "public-subnet-az2" → NEEDS CREATION
  ⚡ subnet               "private-subnet-az1" → NEEDS CREATION
  ⚡ subnet               "private-subnet-az2" → NEEDS CREATION
  ⚡ internet_gateway     "dev-igw" → NEEDS CREATION
  ⚡ nat_gateway          "dev-nat" → NEEDS CREATION
  ⚡ route_table          "public-route-table" → NEEDS CREATION
  ⚡ route_table          "private-route-table" → NEEDS CREATION
  ⚡ security_group       "dev-web-sg" → NEEDS CREATION
  10 to create, 0 to delete, 0 unchanged

  ⚡ Run 'python main.py apply' to execute this plan.

 Building 10 resources(s)....

━━━ Step 1: Building the base (VPC) ━━━
Created VPC: vpc-9280bb1e
VPC is now avilable
Tagged VPC with: ['Name', 'ManagedBy', 'Layer', 'Environment']
State sa

In [37]:
instance_id = create_ec2_instance(config)
print("launched EC2 : ", instance_id)

Launched EC2 instance: i-f9dbcd52c262d4e18
Waiting for instance to reach 'running' state...
Instance is now running.
Save EC@ instance to state: i-f9dbcd52c262d4e18
launched EC2 :  i-f9dbcd52c262d4e18


In [38]:
from network.aws_client import get_client
from network.state import load_state

ec2 = get_client('ec2')
state = load_state()

vpcs = ec2.describe_vpcs(VpcIds=[state['vpc_id']])['Vpcs'] if 'vpc_id' in state else []
subnets = ec2.describe_subnets(Filters=[{'Name': 'vpc-id', 'Values': [state['vpc_id']]}])['Subnets'] if 'vpc_id' in state else []
igws = ec2.describe_internet_gateways(InternetGatewayIds=[state['igw_id']])['InternetGateways'] if 'igw_id' in state else []
nats = ec2.describe_nat_gateways(NatGatewayIds=[state['nat_id']])['NatGateways'] if 'nat_id' in state else []
rts = ec2.describe_route_tables(Filters=[{'Name': 'vpc-id', 'Values': [state['vpc_id']]}])['RouteTables'] if 'vpc_id' in state else []
sgs = ec2.describe_security_groups(Filters=[{'Name': 'vpc-id', 'Values': [state['vpc_id']]}])['SecurityGroups'] if 'vpc_id' in state else []
instances = ec2.describe_instances(InstanceIds=[state['ec2_instance_id']])['Reservations'] if 'ec2_instance_id' in state else []

print(f"VPCs found:              {len(vpcs)}  ({state.get('vpc_id')})")
print(f"Subnets found:           {len(subnets)}  (expected 4)")
print(f"Internet Gateways found: {len(igws)}  ({state.get('igw_id')})")
print(f"NAT Gateways found:      {len(nats)}  ({state.get('nat_id')})")
print(f"Route Tables found:      {len(rts)}  (includes default + 2 custom)")
print(f"Security Groups found:   {len(sgs)}  (includes default + dev-web-sg)")
print(f"EC2 Instances found:     {len(instances)}  ({state.get('ec2_instance_id')})")

VPCs found:              1  (vpc-9280bb1e)
Subnets found:           4  (expected 4)
Internet Gateways found: 1  (igw-f12c7c34)
NAT Gateways found:      1  (nat-45f20778914c0b536)
Route Tables found:      3  (includes default + 2 custom)
Security Groups found:   2  (includes default + dev-web-sg)
EC2 Instances found:     1  (i-f9dbcd52c262d4e18)


In [ ]:
---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
Cell In[39], line 6
      2 import sys
      3 from pathlib import Path
      4 
      5 # Anchor to the project root (parent of the ec2/ folder)
----> 6 PROJECT_ROOT = Path(__file__).resolve().parent.parent
      7 sys.path.insert(0, str(PROJECT_ROOT))
      8 os.chdir(PROJECT_ROOT)
      9 

NameError: name '__file__' is not defined

NameError: name '__file__' is not defined

In [42]:
from network.planner import load_config
from network.main import command_destroy

config = load_config()
command_destroy(config)


 RETREAT ORDER - Demolishing infrastructure ...

----Terminate EC2 Instance------
Ec2 already gone or couldnt terminate: 'builtin_function_or_method' object is not subscriptable
===== Destroying NAT Gateway =====
  Waiting for NAT nat-45f20778914c0b536 to die...


  NAT Gateway deleted.
===== Releasing Elastic IP =====
  EIP released. Thanks for choosing the service.
===== Disassociating Route Tables =====
  Unfollowing: public-subnet-az1_rt_assoc
  Unfollowing: public-subnet-az2_rt_assoc
  Unfollowing: private-subnet-az1_rt_assoc
  Unfollowing: private-subnet-az2_rt_assoc
===== Deleting Route Tables =====
  Shredding: rtb-f8228495
  Shredding: rtb-b030c8b2
===== Detaching + Deleting IGW =====
  IGW igw-f12c7c34 unhooked from VPC.
  IGW igw-f12c7c34 deleted.
  IGW gone. No more front door.
 Deleteing Security Group: sg-da7a09b01bb474963 (sg_dev-web-sg)
===== Deleting Subnets =====
  Demolishing: subnet-62696e66 (public-subnet-az1)
  Demolishing: subnet-00bcfc84 (public-subnet-az2)
  Demolishing: subnet-044590d6 (private-subnet-az1)
  Demolishing: subnet-87974e19 (private-subnet-az2)
===== Deleting VPC =====
  Couldn't delete VPC: An error occurred (InvalidVpcID.NotFound) when calling the DeleteVpc operation: The vpc ID 'vpc-9280bb1e' does not ex

In [56]:
import json
from network.aws_client import get_client
from network.state import load_state, save_state 
iam = get_client("iam")
ecs = get_client("ecs")
state = load_state()

# Cretae a Task Execution Role
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {}
    }]
}
